<a href="https://colab.research.google.com/github/muhyassin09/yasinnn/blob/main/menuju-indonesia-emas-2045/notebooks/00_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Name: Indonesian Municipal Fiscal Analytics Framework
## Module: 00_Data_Acquisition

**Project Pipeline Status:**
- [x] **00_Data_Acquisition.ipynb** *(current)* -> Documents the source publication and how all 8 fiscal ratio tables were extracted from the BPS PDF report.
- [ ] **01_Data_Preprocessing.ipynb** -> Loads the raw fiscal CSV and runs data-quality checks (shape, dtypes, missing values, duplicates).
- [ ] **02_Data_Analysis.ipynb** -> Explores distribution shape/skewness of the 8 fiscal ratios and applies a log1p transform where it helps.
- [ ] **03_Modelling.ipynb** -> Standardizes features, selects k, fits K-Means (k=4), and profiles/visualizes the resulting clusters.
- [ ] **04_Finalizing.ipynb** -> Checks the geographic pattern of clusters, saves the final labeled dataset, and writes the project summary.

---
### 🎯 Module Objective
Document where the raw fiscal data comes from and how it was turned from a BPS PDF publication into a flat, analysis-ready CSV — so the extraction step is reproducible and auditable rather than a black box.

### 📥 Data Ingestion
* **Source Publication:** *Statistik Keuangan Pemerintah Kabupaten/Kota 2023 dan 2024* (BPS-Statistics Indonesia)
  * Publication page: https://www.bps.go.id/id/publication/2024/12/31/6a4becee62edbb7320b6a81e/statistik-keuangan-pemerintah-kabupaten-kota-2023-dan-2024.html
  * Underlying data source: administrative data from the Directorate General of Fiscal Balance (DJPK), Ministry of Finance — realized APBD figures for **2023** (the 2024 column in the same publication is budget/anggaran, not yet realized, and was **not** used, to keep every source in this project on the same 2023 fiscal year).
  * Format: PDF, text-based tables (not scanned), one sub-table per province per indicator, 8 indicators × 37 provinces = 296 sub-tables total.
* **Known extraction hazard:** every page in this publication carries a diagonal BPS watermark (`https://www.bps.go.id`) rendered as individual oversized characters. These interleave with the real table text during naive text extraction (e.g. `Kab. Aceh Besar` reads as `Kab. Aceh Besar o8,59` with a stray `o`). This is filtered out below by dropping characters above a font-size threshold before extracting text — the watermark characters use a visibly larger font (>10.5pt) than the body table text (7–10pt).
* **Current Shape (after extraction):** 508 rows × 10 columns (8 fiscal ratio indicators + `kabupaten_kota` + `provinsi`), covering all 508 kabupaten/kota across 37 provinces (DKI Jakarta has no separate kabupaten/kota APBD and is therefore absent by design, not by error — its kotamadya are administrative subdivisions of one integrated provincial budget).

---



### 🛠️ Environment Setup

In [4]:
import sys
!{sys.executable} -m pip install pdfplumber

import pdfplumber
import pandas as pd
import re

## 1. Locate the publication

1. You can just clone my repositories and locate the raw pdf path.
1. Alternatively, you can go to the publication page linked above and click **Download** and upload them manually.
3. Note the release context in case BPS revises the publication later — this pins the exact version the analysis is based on.



In [9]:
!git clone https://github.com/muhyassin09/yasinnn.git

Cloning into 'yasinnn'...
remote: Enumerating objects: 625, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 625 (delta 33), reused 9 (delta 9), pack-reused 583 (from 1)
Receiving objects: 100% (625/625), 19.27 MiB | 34.93 MiB/s, done.
Resolving deltas: 100% (269/269), done.


In [16]:
PDF_PATH = '/content/yasinnn/menuju-indonesia-emas-2045/data/raw/statistik-keuangan-pemerintah-kabupaten-kota-2023-dan-2024.pdf'
pdf = pdfplumber.open(PDF_PATH)
print('Total pages in publication:', len(pdf.pages))

Total pages in publication: 520


## 2. Strip the watermark before extracting any text

Every page mixes two layers of text: the real table (small font, 7–10pt) and a diagonal watermark made of oversized individual characters (>10.5pt) whose glyphs get interleaved into the extracted line order. Filtering `page.chars` by font size **before** calling `extract_text()` removes the watermark cleanly without touching the real table content — verified on a sample page where filtering took a corrupted line (`Kab. Aceh Besar o8,59`) down to the correct one (`Kab. Aceh Besar 8,59`).


In [22]:
def clean_page_text(page):
    """Drop watermark characters (font size > 10.5pt) before extracting text."""
    filtered = page.filter(lambda obj: obj['object_type'] != 'char' or obj['size'] <= 10.5)
    return filtered.extract_text() or ""

# Double check on one known page (Aceh, Table 4.4 — Decentralization Degree)
sample = clean_page_text(pdf.pages[197])
print(sample[:196])


Tabel 4.4.1 Derajat Desentralisasi Pemerintah Kabupaten/Kota di Provinsi Aceh
(persen), 2023
Table
Decentralization Degree of All Regency/Municipality Governments in
Aceh Province (percent), 2023



## 3. Locate the 8 fiscal indicator sections

The publication's Chapter 4 reports each fiscal ratio as its own run of 37 per-province sub-tables (`Tabel 4.X.1` through `Tabel 4.X.37`, one sub-table per province, kabupaten/kota rows inside each). The 8 indicators used in this analysis, and the page ranges located by scanning for their `Tabel 4.X.Y` headers, are:

| # | Indicator | Column name | Pages |
|---|---|---|---|
| 4.1 | Revenue absorption rate | `tingkat_penyerapan_pendapatan` | 83–119 |
| 4.2 | Expenditure absorption rate | `tingkat_penyerapan_belanja` | 121–157 |
| 4.3 | Tax ratio | `rasio_pajak` | 159–195 |
| 4.4 | Degree of fiscal decentralization | `derajat_desentralisasi_2023` | 197–233 |
| 4.5 | Fiscal independence ratio | `rasio_kemandirian_2023` | 235–271 |
| 4.6 | PAD (own-revenue) effectiveness ratio | `rasio_efektivitas_pad` | 273–309 |
| 4.7 | Tax revenue effectiveness ratio | `rasio_efektivitas_pajak` | 311–347 |
| 4.8 | Expenditure-to-revenue ratio | `rasio_belanja_thd_pendapatan` | 349–385 |

(Two further sections, 4.9 "expenditure realization ratio by type" and 4.10 "expenditure ratio by function", exist in the same chapter but were not used — they break down expenditure by type/function rather than reporting the ratio-level indicators this analysis clusters on.)


In [23]:
def find_indicator_sections(pdf, page_range=(70, 480)):
    """Scan for `Tabel 4.X.Y` headers and return the page span of each top-level section X."""
    section_pages = {}
    for i in range(*page_range):
        txt = clean_page_text(pdf.pages[i])
        if not txt:
            continue
        m = re.search(r'Tabel 4\.(\d+)\.(\d+)\b', txt[:150])
        if m:
            sec = int(m.group(1))
            section_pages.setdefault(sec, []).append(i)
    return {sec: (pages[0], pages[-1]) for sec, pages in section_pages.items()}

sections_found = find_indicator_sections(pdf)
for sec, (start, end) in sorted(sections_found.items()):
    print(f"Section 4.{sec}: pages {start}-{end}")


Section 4.1: pages 83-119
Section 4.2: pages 121-157
Section 4.3: pages 159-195
Section 4.4: pages 197-233
Section 4.5: pages 235-271
Section 4.6: pages 273-309
Section 4.7: pages 311-347
Section 4.8: pages 349-385
Section 4.9: pages 387-423
Section 4.10: pages 425-479


## 4. Extract each indicator table

Each province's sub-table gives one row per kabupaten/kota as `Kab./Kota <name> <value>,<decimal>`. A handful of very low-activity regencies report `~0` (approximately zero) instead of a decimal — e.g. Kab. Pegunungan Arfak and Kab. Pegunungan Bintang for the tax ratio, both remote Papuan regencies with negligible local tax collection. These are recorded as `0.0` rather than dropped, since `~0` is the source's own notation for a real (if tiny) value, not a missing observation.

The province name itself sometimes wraps onto a second line for multi-word provinces (e.g. `... di Provinsi Kalimantan` / `Barat (persen), 2023`) — the parser below joins the first two lines before extracting the province name to handle this.


In [24]:
def parse_indicator_section(pdf, start_page, end_page, column_name):
    """Parse one fiscal indicator's 37 per-province sub-tables into a tidy long-format frame."""
    rows = []
    province = None
    for i in range(start_page, end_page + 1):
        txt = clean_page_text(pdf.pages[i])
        lines = txt.split('\n')

        # Province name can wrap onto line 2 for multi-word provinces (e.g. "Kalimantan" / "Barat")
        header_two_lines = lines[0] + ' ' + lines[1]
        m = re.search(r'di Provinsi (.+?)\s*\(persen\)', header_two_lines)
        if m:
            province = re.sub(r'\s+', ' ', m.group(1)).strip()

        for line in lines:
            line = line.strip()
            row_match = re.match(r'^(Kab\.|Kota)\s+(.+?)\s+([\d]+,\d+)$', line)
            if row_match:
                name = f"{row_match.group(1)} {row_match.group(2)}"
                value = float(row_match.group(3).replace(',', '.'))
                rows.append({'provinsi': province, 'kabupaten_kota': name, column_name: value})
                continue

            approx_zero_match = re.match(r'^(Kab\.|Kota)\s+(.+?)\s+~0$', line)
            if approx_zero_match:
                name = f"{approx_zero_match.group(1)} {approx_zero_match.group(2)}"
                rows.append({'provinsi': province, 'kabupaten_kota': name, column_name: 0.0})

    return pd.DataFrame(rows)


In [25]:
INDICATOR_SECTIONS = {
    1: ('tingkat_penyerapan_pendapatan',      83, 119),
    2: ('tingkat_penyerapan_belanja', 121, 157),
    3: ('rasio_pajak',                   159, 195),
    4: ('derajat_desentralisasi_2023',     197, 233),
    5: ('rasio_kemandirian_2023',   235, 271),
    6: ('rasio_efektivitas_pad',     273, 309),
    7: ('rasio_efektivitas_pajak',     311, 347),
    8: ('rasio_belanja_thd_pendapatan',349, 385),
}

EXPECTED_REGIONS = 508  # official count of Indonesian kabupaten/kota

indicator_dfs = []
for section_id, (column_name, start_page, end_page) in INDICATOR_SECTIONS.items():
    df = parse_indicator_section(pdf, start_page, end_page, column_name)
    status = 'OK' if len(df) == EXPECTED_REGIONS else 'CHECK'
    print(f"4.{section_id} {column_name}: {len(df)} rows [{status}]")
    indicator_dfs.append(df)


4.1 tingkat_penyerapan_pendapatan: 508 rows [OK]
4.2 tingkat_penyerapan_belanja: 508 rows [OK]
4.3 rasio_pajak: 508 rows [OK]
4.4 derajat_desentralisasi_2023: 508 rows [OK]
4.5 rasio_kemandirian_2023: 508 rows [OK]
4.6 rasio_efektivitas_pad: 508 rows [OK]
4.7 rasio_efektivitas_pajak: 508 rows [OK]
4.8 rasio_belanja_thd_pendapatan: 508 rows [OK]


## 5. Consolidate into a single flat file

Each per-indicator table is merged on `['provinsi', 'kabupaten_kota']` — both keys are needed because a handful of kabupaten/kota names repeat across different provinces. An outer join is used deliberately during development so any row that fails to match in one indicator (rather than silently vanishing) shows up as a `NaN` to investigate, instead of being dropped.


In [26]:
master = indicator_dfs[0]
for df in indicator_dfs[1:]:
    master = master.merge(df, on=['provinsi', 'kabupaten_kota'], how='outer')

# Sanity checks before saving
assert master.shape[0] == 508, f"Expected 508 regions, got {master.shape[0]}"
assert master['provinsi'].nunique() == 37, f"Expected 37 provinces, got {master['provinsi'].nunique()}"
assert master.isna().sum().sum() == 0, "Unexpected missing values after merge — investigate before saving"

print('Final merged shape:', master.shape)
print('Provinces covered:', master['provinsi'].nunique())
print(master.isna().sum())


Final merged shape: (508, 10)
Provinces covered: 37
provinsi                         0
kabupaten_kota                   0
tingkat_penyerapan_pendapatan    0
tingkat_penyerapan_belanja       0
rasio_pajak                      0
derajat_desentralisasi_2023      0
rasio_kemandirian_2023           0
rasio_efektivitas_pad            0
rasio_efektivitas_pajak          0
rasio_belanja_thd_pendapatan     0
dtype: int64


## 6. Spot-check against known facts

Before trusting the extraction, a few values are checked against independent, publicly known facts rather than just checking shapes:

* **Kab. Badung (Bali)** should sit at or near the top of `rasio_kemandirian_2023` — its tourism-driven own-revenue (Kuta, Seminyak, the airport) is well known to exceed 100% of central transfers in most years.
* **Remote Papuan regencies** (e.g. Kab. Pegunungan Bintang, Kab. Pegunungan Arfak) should sit at or near the bottom of `rasio_pajak` and `rasio_kemandirian_2023` — consistent with the `~0` tax-ratio notation handled explicitly above.

Both checks passing on values pulled independently from the PDF (not hand-typed) is stronger evidence of a correct extraction than shape checks alone.


In [27]:
print(master.sort_values('rasio_kemandirian_2023', ascending=False).head(5)
      [['provinsi', 'kabupaten_kota', 'rasio_kemandirian_2023']])

print()
print(master.sort_values('rasio_pajak').head(5)
      [['provinsi', 'kabupaten_kota', 'rasio_pajak']])

        provinsi  kabupaten_kota  rasio_kemandirian_2023
23          Bali     Kab. Badung                  695.14
171   Jawa Timur   Kota Surabaya                  150.58
26          Bali    Kab. Gianyar                  141.07
35        Banten  Kab. Tangerang                  123.01
131  Jawa Tengah   Kota Semarang                  112.61

             provinsi           kabupaten_kota  rasio_pajak
323       Papua Barat    Kab. Pegunungan Arfak         0.00
336  Papua Pegunungan  Kab. Pegunungan Bintang         0.00
326  Papua Barat Daya             Kab. Maybrat         0.02
350      Papua Tengah              Kab. Puncak         0.04
351      Papua Tengah         Kab. Puncak Jaya         0.04


## 7. Save the raw extracted dataset

The consolidated table is saved as-is (no cleaning or transformation yet — that happens in `01_Data_Preprocessing.ipynb`), matching what BPS reports, so any downstream data-quality issue can always be traced back to this file.


In [32]:
OUTPUT_PATH = '/content/yasinnn/menuju-indonesia-emas-2045/data/raw/fiscal_indicators_indonesia_2023.csv'
master.to_csv(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)
print('Shape:', master.shape)

Saved: /content/yasinnn/menuju-indonesia-emas-2045/data/raw/fiscal_indicators_indonesia_2023.csv
Shape: (508, 10)


---
### 📤 Data Export & Handoff
* **Output File:** `data/raw/fiscal_indicators_indonesia_2023.csv`
* **Shape:** 508 rows × 10 columns (8 fiscal ratio indicators, kabupaten/kota-level, 2023 realization)
* **Next Destination:** `notebooks/01_Data_Preprocessing.ipynb`
